In [59]:
import os
import random
import unicodedata
import pandas as pd

In [60]:
templates_data = pd.read_csv('personal/sentence_template_starter.csv')
towns_data = pd.read_csv('personal/french_town_start.csv')
generated_data = pd.read_csv('personal/generated_french_town_dataset.csv')


In [61]:
templates_valid = templates_data[templates_data['label'] == 'VALID']
templates_invalid = templates_data[templates_data['label'] == 'INVALID']
both_templates = pd.concat([templates_valid, templates_invalid])
towns = towns_data['nom_ville'].tolist()

In [62]:
generated_data = []
total_data_number = 0

valid_number = random.randrange(1500, 2000)
invalid_number = random.randrange(1500, 2000)
lower_valid = int(valid_number * 0.15)
lower_invalid = int(invalid_number * 0.15)
without_accent = random.randrange(400, 600)

if generated_data:
    os.remove('personal/generated_french_town_dataset.csv')


In [63]:
def generate_sentence(template, departure, arrival, should_lowercase):
    if should_lowercase:
        template = template.lower()
        departure_str = str(departure).lower()
        arrival_str = str(arrival).lower()
    else:
        departure_str = str(departure)
        arrival_str = str(arrival)

    return template.replace('{ville1}', departure_str).replace('{ville2}', arrival_str)


def generate_data_for_label(templates_df, count, lower_count, label, start_id):
    data = []
    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        departure, arrival = random.sample(towns, 2)
        should_lowercase = i < lower_count
        sentence = generate_sentence(row['template'], departure, arrival, should_lowercase)

        if random.random() < 0.1:
            sentence = add_noise_to_sentence(sentence)

        if label == 'VALID':
            data.append({
                'sentence_id': start_id + i,
                'label': label,
                'sentence': sentence,
                'departure': departure,
                'arrival': arrival,
                'template_id': row['template_id']
            })
        else:
            data.append({
                'sentence_id': start_id + i,
                'label': label,
                'sentence': sentence,
                'departure': None,
                'arrival': None,
                'template_id': row['template_id']
            })
    return data


def generate_data_without_accent(templates_df, count, start_id):
    data = []
    lower_count = count // 2

    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        departure, arrival = random.sample(towns, 2)
        should_lowercase = i < lower_count

        sentence = generate_sentence(row['template'], departure, arrival, should_lowercase)
        sentence_no_accent = unicodedata.normalize('NFD', sentence).encode('ascii', 'ignore').decode('utf-8')

        if random.random() < 0.1:
            sentence_no_accent = add_noise_to_sentence(sentence_no_accent)


        if row['label'] == 'VALID':
            data.append({
                'sentence_id': start_id + i,
                'label': row['label'],
                'sentence': sentence_no_accent,
                'departure': departure,
                'arrival': arrival,
                'template_id': row['template_id']
            })
        else:
            data.append({
                'sentence_id': start_id + i,
                'label': row['label'],
                'sentence': sentence_no_accent,
                'departure': None,
                'arrival': None,
                'template_id': row['template_id']
            })

    return data


def add_noise_to_sentence(sentence, noise_level=0.05):
    sentence = list(sentence)

    for i in range(len(sentence)):
        if random.random() < noise_level:
            operator = random.choice(['delete', 'swap', 'duplicate'])

            if operator == 'delete' and len(sentence) > 1:
                sentence[i] = ''
            elif operator == 'swap' and i < len(sentence) - 1:
                sentence[i], sentence[i + 1] = sentence[i + 1], sentence[i]

            elif operator == 'duplicate':
                sentence[i] = sentence[i] * 2

    noisy_sentence = ''.join(sentence)

    if random.random() < noise_level:
        noisy_sentence = ' ' + noisy_sentence
    if random.random() < noise_level:
        noisy_sentence = noisy_sentence + ' '

    return noisy_sentence

In [64]:
generated_data = []

valid_data = generate_data_for_label(
    templates_valid,
    valid_number,
    lower_valid,
    'VALID',
    start_id=1
)

invalid_data = generate_data_for_label(
    templates_invalid,
    invalid_number,
    lower_invalid,
    'INVALID',
    start_id=len(valid_data) + 1
)

accent_data = generate_data_without_accent(
    both_templates,
    without_accent,
    start_id=len(valid_data) + len(invalid_data) + 1
)

generated_data.extend(valid_data)
generated_data.extend(invalid_data)
generated_data.extend(accent_data)

In [65]:
df_generated = pd.DataFrame(generated_data).sample(frac=1)
df_generated.to_csv('personal/generated_french_town_dataset.csv', index=False)
df_generated.head()

,sentence_id,label,sentence,departure,arrival,template_id
2527,2528,INVALID,Mon ami vit à Chalon-sur-Saône et moi à Vienne,NaN,NaN,19
312,313,VALID,Option flexible Lunéville Auch,Lunéville,Auch,210
1399,1400,VALID,Bus direct demain Nantes Mulhouse,Nantes,Mulhouse,282
136,137,VALID,trajet épernay vers brest demain,Épernay,Brest,30
718,719,VALID,Je veux aller de Blois à Perpignan,Blois,Perpignan,1


In [66]:
# APRÈS la génération (après df_generated.to_csv(...))

print("=" * 50)
print("ANALYSE DU DATASET GÉNÉRÉ")
print("=" * 50)

# 1. Taille totale
print(f"\n📊 Nombre total de phrases : {len(df_generated)}")

# 2. Distribution VALID/INVALID
print("\n🏷️ Distribution des labels :")
print(df_generated['label'].value_counts())
print("\nEn pourcentage :")
print(df_generated['label'].value_counts(normalize=True) * 100)

# 3. Longueur des phrases
df_generated['sentence_length'] = df_generated['sentence'].str.len()
print("\n📏 Statistiques de longueur des phrases :")
print(df_generated.groupby('label')['sentence_length'].describe())

# 4. Vérification des None
valid_only = df_generated[df_generated['label'] == 'VALID']
invalid_only = df_generated[df_generated['label'] == 'INVALID']

print(f"\n✅ Phrases VALID avec departure/arrival remplis : {valid_only['departure'].notna().sum()}/{len(valid_only)}")
print(f"❌ Phrases INVALID avec departure/arrival à None : {invalid_only['departure'].isna().sum()}/{len(invalid_only)}")

# 5. Templates utilisés
print("\n📝 Nombre de templates différents utilisés :")
print(f"VALID : {valid_only['template_id'].nunique()}")
print(f"INVALID : {invalid_only['template_id'].nunique()}")

# 6. Top 10 villes
print("\n🏙️ Top 10 villes de départ (VALID) :")
print(valid_only['departure'].value_counts().head(10))

print("\n🏁 Top 10 villes d'arrivée (VALID) :")
print(valid_only['arrival'].value_counts().head(10))

# 7. Exemples aléatoires
print("\n📄 5 exemples VALID aléatoires :")
print(valid_only.sample(5)[['sentence', 'departure', 'arrival']])

print("\n📄 5 exemples INVALID aléatoires :")
print(invalid_only.sample(5)[['sentence', 'departure', 'arrival']])

ANALYSE DU DATASET GÉNÉRÉ

📊 Nombre total de phrases : 4292

🏷️ Distribution des labels :
label
INVALID    2249
VALID      2043
Name: count, dtype: int64

En pourcentage :
label
INVALID    52.399814
VALID      47.600186
Name: proportion, dtype: float64

📏 Statistiques de longueur des phrases :
          count       mean       std   min   25%   50%   75%   max
label                                                             
INVALID  2249.0  25.305469  5.909352   6.0  21.0  25.0  29.0  51.0
VALID    2043.0  38.341654  7.558959  19.0  33.0  37.0  43.0  72.0

✅ Phrases VALID avec departure/arrival remplis : 2043/2043
❌ Phrases INVALID avec departure/arrival à None : 2249/2249

📝 Nombre de templates différents utilisés :
VALID : 160
INVALID : 140

🏙️ Top 10 villes de départ (VALID) :
departure
Évreux              28
Dunkerque           25
Nîmes               17
Concarneau          17
Mâcon               17
Chambéry            17
Tourcoing           17
Mulhouse            16
Thonon-les-Bai